# Data Modeling Task
The goal of this task is to examine the master census metrics JSON file for the CCSVI Dashboard and understand how to transition the existing data into a format usable by an LLM.

### Updates
- master JSON file: `public/data/metrics/census_metrics_by_block_group.json`.
- flattened the nested JSON into a long-form Pandas DataFrame.

### Next Steps
- cleaning and transforming metrics, and preparing the dataset for analysis or LLM integration.

# Inspecting the Master JSON

In [70]:
import json
import pandas as pd

In [71]:
# Loading the master .json file
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

print("Type of raw_data:", type(data))
print("Number of geographic units:", len(data))
print("Sample keys:", list(data.keys())[:5])

Type of raw_data: <class 'dict'>
Number of geographic units: 1158
Sample keys: ['5003', '5004', '5008', '5009', '5011']


In [72]:
# Inspect one example geography
sample_key = list(data.keys())[0]

print("Sample GEOID:", sample_key)
print("\nTop-level keys for this GEOID:")
print(data[sample_key].keys())

Sample GEOID: 5003

Top-level keys for this GEOID:
dict_keys(['type', 'name', 'block_group', 'census_tract', 'county', 'state', 'population', 'metrics'])


In [73]:
# Inspect metrics structure
print("\nMetrics keys:")
print(data[sample_key]["metrics"].keys())

# Look inside one dataset
dataset_name = list(data[sample_key]["metrics"].keys())[0]
print("\nSample dataset:", dataset_name)

print("\nMetric names inside dataset:")
print(data[sample_key]["metrics"][dataset_name].keys())


Metrics keys:
dict_keys(['2022_census_hawaiian_homelands'])

Sample dataset: 2022_census_hawaiian_homelands

Metric names inside dataset:
dict_keys(['Total Population Under 5', 'Total Population Under 18', 'Total Population Over 65', 'Total population SEX Male', 'Total population SEX Female', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race White', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Black or African American', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race American Indian and Alaska Native', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Asian', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Native Hawaiian and Other Pacific Islander', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Some other race', 'Total population RACE AND HISPANIC OR LATINO ORIGIN Two or more races', 'Total population RACE AND HISPANIC OR LATINO ORIGIN Hispanic or Latino origin (of any race)', 'Total population RACE

In [74]:
rows = []

for geoid, area_data in data.items():
    base_info = {
        "geoid": geoid,
        "type": area_data.get("type"),
        "name": area_data.get("name"),
        "block_group": area_data.get("block_group"),
        "census_tract": area_data.get("census_tract"),
        "county": area_data.get("county"),
        "state": area_data.get("state"),
        "population": area_data.get("population")
    }
    
    for dataset_name, metrics in area_data.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            rows.append({
                **base_info,
                "dataset": dataset_name,
                "metric": metric_name,
                "absolute": values.get("absolute"),
                "proportion": values.get("proportion")
            })

df_long = pd.DataFrame(rows)

df_long.head()

,geoid,type,name,block_group,census_tract,county,state,population,dataset,metric,absolute,proportion
0,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Under 5,16.0,0.062257
1,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Under 18,39.0,0.151751
2,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Over 65,21.0,0.081712
3,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total population SEX Male,43.0,0.167315
4,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total population SEX Female,56.0,0.217899


In [67]:
# Check
print(df_long.head())
print("Shape:", df_long.shape)
print("Unique metrics:", df_long["metric"].nunique())
print("Unique geographic units:", df_long["geoid"].nunique())

  geoid               type                                           name  \
0  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
1  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
2  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
3  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
4  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   

  block_group census_tract county state  population  \
0        None         None   None  None       257.0   
1        None         None   None  None       257.0   
2        None         None   None  None       257.0   
3        None         None   None  None       257.0   
4        None         None   None  None       257.0   

                          dataset                       metric  absolute  \
0  2022_census_hawaiian_homelands     Total Population Under 5      16.0   
1  2022_census_hawaiian_homelands    Total Population 